In [ ]:
# ============================================================
# Notebook 9: "Reading the Instruments"
# Regression from the Inside: Seeing the Geometry of Linear Models
# ============================================================

# --- Environment setup (run this cell first) ---
import sys

# Install regression_geometry package if not available
try:
    import regression_geometry
except ImportError:
    print("Installing regression_geometry package...")
    !pip install -q git+https://github.com/carloscotrini/sml_lab26_geometric_regression.git
    print("Done! If you see import errors below, restart the runtime (Runtime → Restart) and run this cell again.")

# --- Standard imports ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import linalg
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import OLSInfluence

from regression_geometry.core import ColumnSpace, Projection, HatMatrix, Ellipsoid
from regression_geometry.core import frisch_waugh_lovell, angle_between, demean
from regression_geometry.data import load_meridian
from regression_geometry.colors import *

# --- Rendering backend toggle ---
INTERACTIVE = True
try:
    from regression_geometry import interactive as viz_mod
    if not viz_mod.AVAILABLE:
        INTERACTIVE = False
except ImportError:
    INTERACTIVE = False

if INTERACTIVE:
    from regression_geometry import interactive as viz
else:
    from regression_geometry import plots as viz

from regression_geometry.scoreboard import GeometricScoreboard
from regression_geometry.exercises import (
    predict_first, diagnose_first, memo, reveal,
    generate_diagnostic_challenge,
)

# --- Reproducibility ---
np.random.seed(42)

> *"The instrument panel doesn't fly the plane — but a pilot who ignores it won't fly long."*

## The Test

Nine notebooks. One theorem. Five instruments on the Scoreboard. Leverage plots, eigenvalue ellipsoids, Relevant Triangles, Frisch-Waugh decompositions, confidence ellipses.

Time to use them.

You're about to see a dataset you've never encountered before. Your job: run a regression, read the instruments, and diagnose every problem — using only the geometric tools you've built. No hints. No guided steps. Just the tools and your judgment.

In [ ]:
# Load the challenge dataset
challenge = generate_diagnostic_challenge(seed=200, difficulty='medium')
X_challenge = challenge['X']
y_challenge = challenge['y']

print(f"Dataset: {challenge['description']}")
print(f"\nObservations: {X_challenge.shape[0]}, Predictors: {X_challenge.shape[1]}")
print(f"Features: {challenge['feature_names']}")

df_challenge = pd.DataFrame(X_challenge, columns=challenge['feature_names'])
df_challenge['y'] = y_challenge
print(f"\n{df_challenge.describe().round(2)}")

In [ ]:
# Fit OLS and display summary + Scoreboard
X_sm = sm.add_constant(X_challenge)
model = sm.OLS(y_challenge, X_sm).fit()
print(model.summary())

cs = ColumnSpace(X_challenge, add_intercept=True)
proj = cs.project(y_challenge)

scoreboard = GeometricScoreboard(
    proj=proj, cs=cs,
    active_gauges=['theta', 'r_squared', 'leverage', 'residual_norm', 'kappa'],
    mode='widget' if INTERACTIVE else 'static'
)
scoreboard.display()

In [ ]:
# 🛑 DIAGNOSE FIRST
diagnose_first(
    summary_text=str(model.summary()),
    question=(
        "(a) Is the column space well-conditioned? (Read κ)\n"
        "(b) Are any observations exerting unusual influence? "
        "(Read tr(H)/n and think about what leverage values to expect)\n"
        "(c) Is the fit reasonable? (Read θ and R²)\n"
        "(d) Are there any coefficients you're suspicious about? Why?"
    )
)

## Eigenvalue Check

The condition number told you something. Now look at the eigenvalue structure directly.

In [ ]:
# Eigenvalue bar chart and condition number
fig = viz.plot_eigenvalue_bar(cs, title="Eigenvalue Structure")

kappa = cs.condition_number()
print(f"Condition number κ = {kappa:.1f}")
print(f"Eigenvalues: {cs.eigenvalues().round(1)}")

If one eigenvalue is much smaller than the others, the column space is elongated — the projection is unstable in that direction. That's what multicollinearity looks like from the inside.

## Leverage and Influence

Leverage is arm length. Influence is arm length times surprise. An observation far from the crowd has long arms — but it only moves the projection if its residual is large too.

In [ ]:
# Leverage, Cook's distance, and influence diagram
hm = HatMatrix(proj.H)
fig_lev = viz.plot_leverage(hm, title="Leverage (hᵢᵢ)")

mse = model.mse_resid
cooks_d = hm.cooks_distance(proj.residuals, mse, cs.p)
fig_cook = viz.plot_cooks_distance(cooks_d, title="Cook's Distance")
fig_inf = viz.plot_influence_diagram(hm, proj.residuals, mse, cs.p,
                                     title="Influence Diagram")

In [ ]:
# 🛑 PREDICT FIRST
predict_first(
    description=(
        "You've seen the Scoreboard, the eigenvalues, and the leverage/influence plots. "
        "You have all the raw diagnostic information."
    ),
    question=(
        "Write down your diagnosis. What problems exist in this regression? "
        "What would you fix? Be specific."
    )
)

## The Relevant Triangles

The Relevant Triangle tells you how much "unique" information each predictor contributes. A short residualized vector means most of that predictor's information is shared with others.

In [ ]:
# Relevant Triangle for each predictor
feature_names = challenge['feature_names']
for j, name in enumerate(feature_names):
    col_idx = j + 1  # +1 because intercept is column 0
    tri = cs.relevant_triangle(y_challenge, col_idx)
    xj_norm = np.linalg.norm(tri['xj_resid'])
    fig = viz.plot_relevant_triangle(cs, y_challenge, col_idx,
                                     title=f"Relevant Triangle: {name}")
    print(f"{name}: ||residualized predictor|| = {xj_norm:.2f}, "
          f"β = {tri['beta_j']:.4f}, SE = {tri['se_j']:.4f}")

In [ ]:
# FWL for the most collinear predictor
norms = []
for j in range(len(feature_names)):
    tri = cs.relevant_triangle(y_challenge, j + 1)
    norms.append(np.linalg.norm(tri['xj_resid']))
suspect_j = int(np.argmin(norms)) + 1
suspect_name = feature_names[suspect_j - 1]

print(f"Most collinear predictor: {suspect_name} "
      f"(shortest residualized vector: {norms[suspect_j-1]:.2f})")

fig_fwl = viz.plot_fwl_decomposition(
    cs.X, y_challenge, suspect_j,
    title=f"FWL Decomposition: {suspect_name}"
)
fig_av = viz.plot_added_variable(
    cs.X, y_challenge, suspect_j,
    title=f"Added Variable Plot: {suspect_name}"
)

The added variable plot shows what's left of the relationship after the shared information has been peeled away. A noisy, weak slope here means the predictor doesn't add much beyond what the other predictors already provide.

## The Reveal

Here are the actual problems baked into this dataset. Compare to your diagnosis.

In [ ]:
# The ground truth
print("TRUE ISSUES IN THIS DATASET")
print("=" * 50)
for i, issue in enumerate(challenge['true_issues'], 1):
    print(f"\n{i}. {issue}")
print(f"\nHINTS (for next time):")
for hint in challenge['hints']:
    print(f"  • {hint}")

reveal(
    "Two issues: multicollinearity between experience and tenure "
    "(the eigenvalue structure flagged this), and one influential "
    "observation with an unusual predictor combination. "
    "The κ gauge and the influence diagram were the key instruments."
)

## A Harder Challenge

That was the medium difficulty. Now try a hard one. Same process: Scoreboard, eigenvalues, leverage, Relevant Triangles. Diagnose everything.

In [ ]:
# Hard challenge: all diagnostics look healthy
hard = generate_diagnostic_challenge(seed=300, difficulty='hard')
X_hard, y_hard = hard['X'], hard['y']
print(f"Dataset: {hard['description']}")
print(f"Features: {hard['feature_names']}")

X_hard_sm = sm.add_constant(X_hard)
model_hard = sm.OLS(y_hard, X_hard_sm).fit()
print(model_hard.summary())

cs_hard = ColumnSpace(X_hard, add_intercept=True)
proj_hard = cs_hard.project(y_hard)
sb_hard = GeometricScoreboard(
    proj=proj_hard, cs=cs_hard,
    active_gauges=['theta', 'r_squared', 'leverage', 'residual_norm', 'kappa'],
    mode='widget' if INTERACTIVE else 'static'
)
sb_hard.display()

In [ ]:
# 🛑 DIAGNOSE FIRST
diagnose_first(
    summary_text=str(model_hard.summary()),
    question=(
        "The Scoreboard is green. The eigenvalues are well-spread. "
        "No extreme leverage points. Is this regression trustworthy? "
        "If not, what's wrong — and how do you know if the geometry can't tell you?"
    )
)

In [ ]:
# The hard reveal
print("TRUE ISSUES IN THIS DATASET")
print("=" * 50)
for i, issue in enumerate(hard['true_issues'], 1):
    print(f"\n{i}. {issue}")
print()
for hint in hard['hints']:
    print(f"  • {hint}")

The geometry couldn't warn you. This is the Notebook 6 lesson in action: a well-conditioned projection onto the wrong column space. The Scoreboard has no gauge for "right wall."

You needed domain knowledge — the cover story mentioned that tutoring programs are more common in affluent districts, but household income wasn't collected. Socioeconomic status drives both tutoring access and test scores. The coefficient on tutoring hours is biased upward because it absorbs the effect of the omitted confounder.

## The Rosetta Stone

You've now used every geometric tool in the course on live data. Here's the reference that maps them all to the code you'll actually type on Tuesday morning.

In [ ]:
# Display the Rosetta Stone cheat sheet
from regression_geometry.cheatsheet import display_cheatsheet
display_cheatsheet()

In [ ]:
# Save and download the cheat sheet
from regression_geometry.cheatsheet import generate_cheatsheet_html

html = generate_cheatsheet_html(output_path='regression_geometry_cheatsheet.html')
print("Cheat sheet saved to: regression_geometry_cheatsheet.html")
print("Open in a browser and print (Ctrl+P / Cmd+P) for a wall reference.")

try:
    from google.colab import files
    files.download('regression_geometry_cheatsheet.html')
except ImportError:
    pass  # not in Colab

Print it. Pin it to your monitor. It's the artifact you take from this course back to your desk.

---

### ✍️ The Memo

Write a memo to your future self. Six months from now, you're staring at a regression output that looks wrong but you can't figure out why. What are the first five things you check — in order — and what does each one tell you?

**Banned words:** *column space*, *projection*, *eigenvalue*

Write it in language you'd actually use at work.

---

In [ ]:
# YOUR MEMO:

In [ ]:
# Geometric Scoreboard — all 5 gauges, Meridian full model
df = load_meridian()
y_m = df['salary'].values
X_m = df[['experience', 'education', 'performance']].values
cs_m = ColumnSpace(X_m, add_intercept=True)
proj_m = cs_m.project(y_m)

sb_final = GeometricScoreboard(
    proj=proj_m, cs=cs_m,
    active_gauges=['theta', 'r_squared', 'leverage', 'residual_norm', 'kappa'],
    mode='widget' if INTERACTIVE else 'static'
)
sb_final.display()

You've been reading these instruments since Notebook 1, when four of the five were grayed out. Now you can read all five at a glance.

---

### Summary

**What you learned:**
- You diagnosed unfamiliar regressions using the geometric tools: the Scoreboard, eigenvalue structure, leverage and influence, Relevant Triangles, and FWL decomposition
- Some problems the geometry caught — multicollinearity, influential observations
- One problem it couldn't catch — omitted variable bias requires causal reasoning beyond the geometric framework

**Key geometric insight:** ***Reading the instruments is a skill. The Scoreboard diagnoses geometry. Your brain diagnoses causality. You need both.***

### Next

Notebook 10 — the last notebook. What can the geometry see? What can't it see? And where do you go from here?

---